In [1]:
import pandas as pd

df = pd.read_csv('raw_data.csv')

C:\Users\busaji\AppData\Local\Temp\ipykernel_39144\3790256753.py:3: DtypeWarning: Columns (0: PRES Response Number, 1: Study CPMS, 2: RRDN match using Trust ODS code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('raw_data.csv')


In [7]:

df['Text'].fillna(value="None", inplace=True)
#study_df2['StudyResearchSummary'] = study_df2['StudyResearchSummary'].encode('utf-8')

for index, row in df.iterrows():
    row['Text'] = row['Text'].encode('utf-8')




C:\Users\busaji\AppData\Local\Temp\ipykernel_39144\591362491.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['Text'].fillna(value="None", inplace=True)


AttributeError: 'float' object has no attribute 'encode'

# Feature Extraction using TfidfVectorizer
[vectorizer error on nan fixed using this solution](https://stackoverflow.com/questions/39303912/tfidfvectorizer-in-scikit-learn-valueerror-np-nan-is-an-invalid-document)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from time import time
import numpy as np

vectorizer = TfidfVectorizer(
    max_df=0.5,
    min_df=5,
    stop_words="english",
    # encoding = "utf-8", decode_error="ignore"
)
t0 = time()
X_tfidf = vectorizer.fit_transform(df['Text'].values.astype('U'))

print(f"vectorization done in {time() - t0:.3f} s")
print(f"n_samples: {X_tfidf.shape[0]}, n_features: {X_tfidf.shape[1]}")
print(f"{X_tfidf.nnz / np.prod(X_tfidf.shape):.3f}")

vectorization done in 1.245 s
n_samples: 157426, n_features: 8286


# Dimensionality Reduction

In [12]:
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer

lsa = make_pipeline(TruncatedSVD(n_components=100), Normalizer(copy=False))
t0 = time()
X_lsa = lsa.fit_transform(X_tfidf)
explained_variance = lsa[0].explained_variance_ratio_.sum()

print(f"LSA done in {time() - t0:.3f} s")
print(f"Explained variance of the SVD step: {explained_variance * 100:.1f}%")

LSA done in 3.492 s
Explained variance of the SVD step: 42.4%


In [ ]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
# Source - https://stackoverflow.com/a/62613199
# Posted by gurezende, modified by community. See post 'Timeline' for change history
# Retrieved 2026-04-29, License - CC BY-SA 4.0

#Creating models for k from 2 to 14
inertia = []
for k in range(2,50):
    model = KMeans(n_clusters=k, random_state=12).fit(X_lsa)
    inertia.append(model.inertia_)

#Plotting the inertia of the models    
k_values = range(2,50)
plt.plot(k_values, inertia, 'o-')
plt.xlabel('Number of Clusters')
plt.ylabel('Inertia')




# for seed in range(5):
#     kmeans = KMeans(
#         n_clusters=true_k,
#         max_iter=100,
#         n_init=1,
#         random_state=seed,
#     ).fit(X_tfidf)
#     cluster_ids, cluster_sizes = np.unique(kmeans.labels_, return_counts=True)
#     print(f"Number of elements assigned to each cluster: {cluster_sizes}")
# print()
# print(
#     "True number of documents in each category according to the class labels: "
#     f"{category_sizes}"
# )


# kmeans = KMeans(
#     n_clusters=true_k,
#     max_iter=100,
#     n_init=5,
# )

# fit_and_evaluate(kmeans, X_tfidf, name="KMeans\non tf-idf vectors")